***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [1. 利用干涉阵开展射电科学](1_0_introduction.ipynb)
    * 上一节： [1.8 天文射电源](1_8_astronomical_radio_sources.ipynb)
    * 下一节： [1.10 单碟望远镜的局限与价值](1_10_limits_of_single_dishes.ipynb)
***


导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
HTML('../style/course.css') #apply general CSS

导入本节所需的专用模块:

In [ ]:
from IPython.display import display
try:
    from ipywidgets import interact
except ImportError:
    def interact(func, **kwargs):
        defaults = {}
        for key, value in kwargs.items():
            if isinstance(value, tuple):
                defaults[key] = value[0]
            else:
                defaults[key] = value
        return func(**defaults)


In [ ]:
HTML('../style/code_toggle.html')

## 1.9 干涉测量：从双缝到孔径合成

到本节为止，我们已经知道射电科学需要高角分辨率，也知道不同辐射机制和射电源会对观测提出非常不同的要求。接下来的问题是：干涉阵到底在测什么？为什么几台甚至几十台分散的天线，能够组合成一台“更大”的望远镜？在正式进入可见度空间和傅里叶成像之前，本节先从最经典的双缝实验出发，为这些问题建立物理直觉。

本节要建立三个后续会反复出现的核心观念。干涉测量本质上是在比较不同路径上的相位关系；一条基线只提供关于天空的一部分信息，而不是完整图像；只有通过多条基线、地球自转和孔径合成，才有可能逐步重建天空亮度分布。把这三点先建立起来，后续可见度、$uv$ 采样和成像反演就会自然许多。


### 1.9.1 为什么先从双缝开始

干涉测量的历史起点，通常追溯到 1801 年托马斯·杨提出的双缝实验。这个实验最初的目的并不是成像，而是证明光具有波动性。但对射电干涉测量教学来说，它还有另一重价值：它把“相位差如何转化为可测量强度变化”这件事，直观地展示了出来。

<img src="figures/double_slit_schematic.png" width="80%"/>

**图 1.9.1**：双缝实验的路径差与屏上强度条纹。图为本项目生成的概念示意，不按实验装置比例绘制。

在双缝实验中，两条路径上的电场在屏幕处相加。若两路单色波写成复振幅形式，则有

$$E = E_1 + E_2 = A e^{i\phi} + A e^{i(\phi-\phi_0)}$$

其中 $\phi_0$ 对应两条路径造成的相位差。探测器真正测到的不是瞬时电场本身，而是时间平均后的强度，也就是 $EE^*$。把它展开后可得

$$EE^* = 2A^2 + 2A^2\cos\phi_0$$

因此，干涉条纹的本质并不是“光在屏幕上变花了”，而是相位差被转写成了可测量的亮度起伏。对后续学习而言，这一点至关重要，因为现代射电干涉仪虽然不再真的在屏幕上看条纹，但它们仍然在测量同一件事：不同阵元接收到的电场之间的相关与相位关系。


### 1.9.2 一个双缝干涉玩具模拟器

下面这个 Python 函数并不是严格的光学传播模拟，而是一个概念演示模型。它的作用是把“路径差如何改变条纹”和“源结构如何影响干涉图样”这两件事可视化。它不应被理解为精确的实验设计软件，而应被理解为建立干涉直觉的简化图像。


In [ ]:
def double_slit (p0=[0],a0=[1],baseline=1,d1=5,d2=5,wavelength=.1,maxint=None):
    """Renders a toy dual-slit experiment.
    'p0' is a list or array of source positions (drawn along the vertical axis)
    'a0' is an array of source intensities 
    'baseline' is the distance between the slits 两狭缝间的距离为‘baseline’
    'd1' and 'd2' are distances between source and plate and plate and screen ‘d1’‘d2’分别是源和遮光板，遮光板跟像屏间的距离
    'wavelength' is wavelength
    'maxint' is the maximum intensity scale use to render the fringe pattern. If None, the pattern
       is auto-scaled. Maxint is useful if you want to render fringes from multiple invocations
       of double_slit() into the same intensity scale, i.e. for comparison.
    """
    ## setup figure and axes
    plt.figure(figsize=(20, 5))
    plt.axes(frameon=False)
    plt.xlim(-d1-.1, d2+2) and plt.ylim(-1, 1)
    plt.xticks([]) and plt.yticks([])
    plt.axhline(0, ls=':')
    baseline /= 2.
    ## draw representation of slits
    plt.arrow(0, 1,0, baseline-1, lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0,-1,0, 1-baseline, lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0, 0,0,  baseline,  lw=0, width=.1, head_width=.1, length_includes_head=True)
    plt.arrow(0, 0,0, -baseline,  lw=0, width=.1, head_width=.1, length_includes_head=True)
    ## draw representation of lightpath from slits to centre of screen
    plt.arrow(0, baseline,d2,-baseline, length_includes_head=True)
    plt.arrow(0,-baseline,d2, baseline, length_includes_head=True)
    ## draw representation of sinewave from the central position
    xw = np.arange(-d1, -d1+(d1+d2)/4, .01)
    yw = np.sin(2*np.pi*xw/wavelength)*.1 + (p0[0]+p0[-1])/2
    plt.plot(xw,yw,'b')
    ## 'xs' is a vector of x cordinates on the screen
    ## and we accumulate the interference pattern for each source into 'pattern'
    xs = np.arange(-1, 1, .01) 
    pattern = 0
    total_intensity = 0
    ## compute contribution to pattern from each source position p
    for p,a in np.broadcast(p0,a0):
        plt.plot(-d1, p, marker='o', ms=10, mfc='red', mew=0)
        total_intensity += a
        if p == p0[0] or p == p0[-1]:
            plt.arrow(-d1, p, d1, baseline-p, length_includes_head=True)
            plt.arrow(-d1, p, d1,-baseline-p, length_includes_head=True)
        # compute the two pathlenghts
        path1 = np.sqrt(d1**2 + (p-baseline)**2) + np.sqrt(d2**2 + (xs-baseline)**2)
        path2 = np.sqrt(d1**2 + (p+baseline)**2) + np.sqrt(d2**2 + (xs+baseline)**2)
        diff = path1 - path2
        # caccumulate interference pattern from this source
        pattern = pattern + a*np.cos(2*np.pi*diff/wavelength) 
    maxint = maxint or total_intensity
    # add fake axis to interference pattern just to make it a "wide" image
    pattern_image = pattern[:,np.newaxis] + np.zeros(10)[np.newaxis,:]
    plt.imshow(pattern_image, extent=(d2,d2+1,-1,1), cmap=plt.gray(), vmin=-maxint, vmax=maxint)
    # make a plot of the interference pattern
    plt.plot(d2+1.5+pattern/(maxint*2), xs, 'r')
    plt.show()
# show pattern for one source at 0
double_slit(p0=[0])

`double_slit` 函数会画出一个简化的双缝装置。红点表示源的位置，蓝色正弦波只是在示意波长，黑线表示光路，而右侧灰度条带和红色曲线表示最终条纹图样及其截面。这里真正关心的不是图像画得多漂亮，而是条纹随基线、波长和源结构变化时的规律。

<div class=warn>
<b>注意：</b> 这个模拟器是 1 维、远场近似下的概念演示，没有严格处理衍射传播和真实光学系统。后面真实干涉阵中的许多概念，只能把它当成直觉出发点，而不能直接把图上的每个细节照搬到工程系统中。
</div>


### 1.9.3 基线与波长为什么决定分辨能力

先只看一个点源。调节基线 $B$ 和波长 $\lambda$ 后，可以看到：基线越长，条纹越密；波长越短，条纹也越密。这正是为什么干涉测量的角分辨率按 $\lambda/B$ 缩放。


In [ ]:
interact(lambda baseline,wavelength:double_slit(p0=[0],baseline=baseline,wavelength=wavelength),
                baseline=(0.1,2,.01),wavelength=(.05,.2,.01)) and None

不过，这里必须先建立一个很重要的区分。条纹变密并不等于“已经得到一张更清楚的图像”，它只意味着系统对源位置和源结构变化变得更敏感了。换句话说，基线控制的是干涉仪对某一空间尺度的响应，而不是直接控制一张现成图像的像素大小。真正的成像，还需要更多基线和更系统的重建过程。


### 1.9.4 从双缝实验到可测量的天文信息

一旦把双缝装置看成测量设备，而不只是证明波动性的演示装置，就会发现：它并不只是告诉我们“有条纹”，而是已经能够反推出源的位置和结构信息。这正是干涉测量成为天文工具的起点。

#### 1.9.4.1 位置测量：相位首先告诉我们源在哪里

下面保持源为点源，只改变它相对于光轴的位置。干涉图样的整体相位会随源位置改变，而且这种变化对长基线尤其敏感。


In [ ]:
interact(lambda position,baseline,wavelength:double_slit(p0=[position],baseline=baseline,wavelength=wavelength),
               position=(-1,1,.01),baseline=(0.1,2,.01),wavelength=(.05,.2,.01)) and None

这就是为什么在最简单的情形下，我们常说：**可见度相位最直接地编码源位置。** 后面在复可见度语言里，这会表现为一个随基线变化的复相位因子。长基线对位置变化更敏感，因此给出更高的角分辨能力；但它也会带来另一个问题：相位是周期性的，于是位置测量可能出现多解。


In [ ]:
double_slit([0],baseline=1.5,wavelength=0.1)
double_slit([0.69],baseline=1.5,wavelength=0.1)

上面两组位置完全不同的源，在同一条长基线上却可能给出几乎相同的干涉图样。这说明一条基线本身并不能唯一确定源的位置，尤其当基线很长时，条纹周期性会引入模糊性。


In [ ]:
double_slit([0],baseline=0.5,wavelength=0.1)
double_slit([0.69],baseline=0.5,wavelength=0.1)

较短基线虽然分辨力较低，却能帮助消除这种歧义。因此，现代干涉阵并不会只追求最长基线，而是需要一组长短不同、方向也不同的基线共同工作。这里已经可以先提前记住一句后面会反复出现的话：**在每一个时间、频率通道和偏振相关乘积上，一条基线只返回一个复数，而不是一张图像。** 时间积分、通道平均和多偏振产品会产生一组样本，但每个样本仍只约束天空的一个复数投影。要想真正恢复天空结构，必须把很多基线的信息拼起来。


#### 1.9.4.2 尺度测量：振幅告诉我们源有多“展”

相位主要反映位置，而干涉图样的对比度或振幅则更直接反映源结构。先加入第二个点源，看看两个源的干涉图样如何叠加。


In [ ]:
interact(lambda position,intensity,baseline,wavelength:
            double_slit(p0=[0,position],a0=[1,intensity],baseline=baseline,wavelength=wavelength),
         position=(-1,1,.01),intensity=(.2,1,.01),baseline=(0.1,2,.01),wavelength=(.01,.2,.01)) and None

不同位置和强度组合会让条纹对比度减弱，甚至在某些基线上几乎完全抵消。这就是可见度振幅对源结构敏感的最直接表现。对于双点源来说，某条基线上的对比度，反映的是这条基线能否“分辨出”两个分量之间的分离。


In [ ]:
double_slit(p0=[0,0.25],baseline=1,wavelength=0.1)
double_slit(p0=[0,0.25],baseline=1.5,wavelength=0.1)

更一般地，如果源不是几个离散点，而是一块连续展源，那么不同位置上发出的波前会彼此洗出，条纹振幅会随着源尺度增大而减弱。下面让源逐渐变宽，可以直观看到这种效应。


In [ ]:
interact(lambda extent,baseline,wavelength:
             double_slit(p0=np.arange(-extent,extent+.01,.01),baseline=baseline,wavelength=wavelength),
         extent=(0,1,.01),baseline=(0.1,2,.01),wavelength=(.01,.2,.01)) and None

这一步对后续干涉测量极其重要，因为它说明：**长基线之所以“看不见”大尺度结构，并不是那些结构不存在，而是它们在这条基线上已经被分辨掉了。** 也正因此，干涉阵的不同基线长度对应对不同空间尺度的敏感性，而不是简单地“都在看同一张图的不同像素”。


In [ ]:
double_slit(p0=[0],baseline=1,wavelength=0.1)
double_slit(p0=np.arange(-0.2,.21,.01),baseline=1,wavelength=0.1)

在最简单的两元干涉仪语言里，我们可以把一条基线的测量结果概括为一个复可见度

$$V(B)=|V(B)|e^{i\phi(B)}$$

其中 $|V|$ 近似反映源在这条基线对应尺度上的结构响应，$\phi$ 近似反映相对于参考方向的位置偏移。后面第 4 章会把这一点推广到二维基线和完整天空亮度分布，并给出严格的傅里叶关系。

顺便说一句，这也是“可见度”一词的历史来源。历史上的条纹可见度通常指无量纲对比度 $\mathcal{V}=(I_{\max}-I_{\min})/(I_{\max}+I_{\min})$。现代射电干涉测量中，经过通量标定的复可见度通常保留相关通量密度的量纲（常用 Jy），并同时包含振幅与相位；只有再用总强度或自相关作归一化，才得到与无量纲相干度更接近的量。二者相关，但不能把“0.5 的条纹对比度”和“0.5 Jy 的可见度振幅”当成同一个物理量。


#### 1.9.4.3 仪器几何也会被写进干涉图样

干涉图样不仅对源敏感，也对仪器自身的几何结构敏感。改变狭缝、屏幕或光路长度，条纹形态就会随之变化。这提醒我们：真实干涉测量里，若不准确掌握阵元位置、时延和系统响应，观测到的相位和振幅就会混合“天体信息”和“仪器信息”。这正是后面定标章节存在的根本原因。


In [ ]:
interact(lambda d1,d2,position,extent: double_slit(p0=np.arange(position-extent,position+extent+.01,.01),d1=d1,d2=d2),
         d1=(1,5,.1),d2=(1,5,.1),
         position=(-1,1,.01),extent=(0,1,.01)) and None

因此，干涉测量不仅是研究天体的工具，也可以反过来高精度约束仪器几何。大地测量 VLBI 用已知射电源测大陆漂移，LIGO 用激光干涉测微小时空扰动，本质上都依赖这种“相位对几何极其敏感”的性质。


### 1.9.5 从实验装置到真实天文干涉仪

真正的天文干涉仪当然不会把两条狭缝和一块屏幕装进一个大盒子里。对天文学而言，基线往往很长，波前来自无限远处，系统也必须可转向、可定时、可记录。因此，历史上真实干涉仪的发展，本质上是在寻找一种更适合天文观测的“路径比较”实现方式。

在光学波段，迈克尔逊恒星干涉仪仍然保留了把两路光重新汇聚到同一探测器上的思路：

<img src="figures/stellar_interferometer_schematic.png" width="80%"/>

**图 1.9.2**：迈克尔逊恒星干涉仪的两端收集镜、可调基线与中央合束器。图为本项目生成的概念示意。

最著名的早期成果之一，是 1920 年 Michelson 和 Pease 用威尔逊山 100 英寸胡克望远镜测量参宿四角直径。其装置可概括为在望远镜顶部架设一条可调基线，再把两端光束送入共同焦点：

<img src="figures/hooker_interferometer_schematic.png" width="80%"/>

**图 1.9.3**：Michelson--Pease 实验在大型望远镜顶部设置恒星干涉基线的概念配置。图为本项目生成的示意，并非历史装置复原图。

在现代光学干涉里，这一路线延续到甚大望远镜干涉仪等系统：

<img src="figures/optical_interferometer_array.png" width="90%"/>

**图 1.9.4**：现代光学长基线干涉阵的概念结构。分离的望远镜把光束送入延迟线和合束器，在探测前补偿几何光程差；图为本项目生成，不复原某一具体设施。

而在射电波段，问题的解法更直接。天线接收到的电场可以先被放大、下变频和数字化，然后再通过电子学和相关器处理。这意味着我们不必真正把两路波在同一块“屏幕”上光学叠加，而是可以先独立记录各阵元信号，再在后端计算它们的相关。1940 年代澳大利亚的海崖干涉仪就是早期实例之一：

<img src="figures/sea_cliff_schematic.png" width="80%"/>

**图 1.9.5**：海崖干涉仪利用天体信号的直达路径与海面反射路径形成等效双通道。图为本项目生成的概念示意。

这种能力最终发展成真正的射电阵列。今天的 JVLA、MeerKAT 等系统，本质上都是把许多阵元的电场采样送入相关器，实时得到多条基线的复可见度：

<img src="figures/connected_array_correlator.png" width="85%"/>

**图 1.9.6**：连接阵把各阵元的独立电压流送入延迟、通道化与相关处理，输出按天线对组织的复可见度。图为本项目生成。

<img src="figures/array_core_and_arms.png" width="70%"/>

**图 1.9.7**：一种兼顾扩展结构灵敏度与角分辨率的示意布局：紧凑核心提供短基线，外延阵元提供长基线。它用于说明设计原则，不对应某个阵列的精确台站坐标。

这里还值得区分两类语言。双缝和迈克尔逊更像**加法式干涉仪**：两路信号先相加，再由探测器感受总强度中的干涉项。现代射电系统则通常是**乘法式干涉仪**：各阵元信号独立记录后，相关器直接计算交叉项 $\langle E_1E_2^*\rangle$。若来自方向 $\hat{s}$ 的平面波到达两天线之间存在几何时延 $\tau_g=\mathbf{b}\cdot\hat{s}/c$，相关器必须通过延迟模型把这种已知几何相位跟真正的天体结构信息区分开来。正因为如此，射电干涉仪天然输出的是复可见度，而不是一张肉眼可见的条纹照片；定标的任务，也正是把仪器增益、时钟、传播介质和几何误差从这个复数测量中分离出去。


### 1.9.6 孔径合成：为什么多条基线能够成像

干涉测量最初常被用于某种定向实验，例如测恒星直径、测源位置或测几何结构。真正让它变成通用成像方法的，是孔径合成思想。其核心并不神秘：既然一条基线只给出天空的一部分信息，那么只要收集足够多、方向和长度不同的基线，就可以把这些碎片信息重新合成为天空图像。

先看下面三种不同的“天空”。在某一条特定基线上，它们可能表现得非常相似，甚至无法区分：


In [ ]:
double_slit(p0=[0], a0=[0.4], maxint=2)
double_slit(p0=[0,0.25], a0=[1, 0.6], maxint=2)
double_slit(p0=np.arange(-0.2,.21,.01), a0=.05, maxint=2)

但一旦换一条基线，响应就会立刻分开：


In [ ]:
double_slit(p0=[0], a0=[0.4], baseline=0.5, maxint=2)
double_slit(p0=[0,0.25], a0=[1, 0.6], baseline=0.5, maxint=2)
double_slit(p0=np.arange(-0.2,.21,.01), a0=.05,  baseline=0.5, maxint=2)

这就是孔径合成最重要的物理直觉：不同基线在采样不同空间尺度和不同方向的信息。用后面第 4 章会正式介绍的语言说，每一条基线都在测天空亮度分布的一个复数投影。忽略偏振和其他方向依赖效应时，完整的标量宽视场测量方程可先写成

$$V(u,v,w)=\iint\frac{A(l,m)I(l,m)}{n}\exp\{-2\pi i[ul+vm+w(n-1)]\}\,dl\,dm,$$

其中 $n=\sqrt{1-l^2-m^2}$，$(u,v,w)$ 是以波长为单位的基线分量，$A(l,m)$ 是阵元主波束。该式已经表明：宽视场时天空曲率带来 $1/n$ 和非线性的 $w(n-1)$ 相位，主波束也会调制真实天空。

只有在小视场、$n\simeq1$、主波束在目标区域近似常数，并且 $w(n-1)$ 可忽略或已经校正时，关系才约化为二维傅里叶形式

$$V(u,v) = \iint I(l,m)\,e^{-2\pi i(ul+vm)}\,dl\,dm$$

这里 $(u,v)$ 是以波长为单位的基线坐标，$(l,m)$ 是天空方向余弦。这个式子此处不需要完整推导，但它揭示了整件事的结构：**可见度不是图像本身，而是图像的傅里叶信息。** 因此，成像问题本质上是一个采样与反演问题，而不只是“把很多小望远镜的信号加起来”。

随着地球自转、阵元数量增加以及多构型观测的引入，$uv$ 采样会逐步变得丰富，使干涉阵从“测一个物理量的装置”变成“可以恢复整幅天空亮度分布的成像系统”。但采样永远不会无限完整，因此真实图像还会受到点扩散函数、权重选择、缺短间距和噪声的影响。这些问题正是后续成像与去卷积章节要系统处理的内容。


### 1.9.7 本节小结

本节从双缝实验出发，建立了干涉测量的基本直觉：不同路径上的相位差可以转化成可测量的强度或相关信号。对天文干涉仪而言，一条基线在每个时间、频率和偏振样本上返回一个复可见度，而不是一张完整图像；在简单情形下，可见度相位主要编码源位置，可见度振幅主要反映源结构和尺度。历史条纹对比度是无量纲归一化量，现代标定复可见度则通常以 Jy 表示相关通量。

真实干涉图样同样对仪器几何、时延和系统响应敏感，因此干涉测量不可能脱离定标。孔径合成之所以能成像，是因为多条基线共同采样了天空亮度分布的不同傅里叶分量。带着这些直觉进入下一节，就可以更清楚地理解：为什么单碟望远镜仍然重要，却不足以独自支撑现代高分辨率射电科学；而干涉阵又是以什么代价换来了这种分辨能力。


### 附录：重现迈克尔逊干涉仪

下面保留一个更接近真实干涉仪布局的玩具模型。它把光源放到无限远，并把双缝改造成迈克尔逊式光路，用来把前面的直觉和真实天文干涉仪联系起来。附录不是本章主线的必要部分，但它适合用来进一步理解“条纹图样”“可见度”与“有限基线长度”的关系。


In [ ]:
def michelson (p0=[0],a0=[1],baseline=50,maxbaseline=100,extent=0,d1=9,d2=1,d3=.2,wavelength=.1,fov=5,maxint=None):
    """Renders a toy Michelson interferometer with an infinitely distant (astronomical) source
    'p0' is a list or array of source positions (as angles, in degrees).
    'a0' is an array of source intensities 
    'extent' are source extents, in degrees
    'baseline' is the baseline, in lambdas
    'maxbaseline' is the max baseline to which the plot is scaled 
    'd1' is the plotted distance between the "sky" and the interferometer arms
    'd2' is the plotted distance between arms and screen, in plot units
    'd3' is the plotted distance between inner mirrors, in plot units
    'fov' is the notionally rendered field of view radius (in degrees)
    'wavelength' is wavelength, used for scale
    'maxint' is the maximum intensity scale use to render the fringe pattern. If None, the pattern
       is auto-scaled. Maxint is useful if you want to render fringes from multiple invocations
       of michelson() into the same intensity scale, i.e. for comparison.
    """
    ## setup figure and axes
    plt.figure(figsize=(20, 5))
    plt.axes(frameon=False)
    plt.xlim(-d1-.1, d2+2) and plt.ylim(-1, 1)
    plt.xticks([]) 
    # label Y axis with degrees
    yt,ytlab = plt.yticks()
    plt.yticks(yt,["-%g"%(float(y)*fov) for y in yt]) 
    plt.ylabel("Angle of Arrival (degrees)")
    plt.axhline(0, ls=':')
    ## draw representation of arms and light path
    maxbaseline = max(maxbaseline,baseline)
    bl2 = baseline/float(maxbaseline)    # coordinate of half a baseline, in plot units
    plt.plot([0,0],[-bl2,bl2], 'o', ms=10)
    plt.plot([0,d2/2.,d2/2.,d2],[-bl2,-bl2,-d3/2.,0],'-k')
    plt.plot([0,d2/2.,d2/2.,d2],[ bl2, bl2, d3/2.,0],'-k')
    plt.text(0, 0, r'$b=%d\lambda$' % baseline, ha='right', va='bottom', size='xx-large')
    ## draw representation of sinewave from the central position
    if isinstance(p0,(int,float)):
        p0 = [p0]
    xw = np.arange(-d1, -d1+(d1+d2)/4, .01)
    yw = np.sin(2*np.pi*xw/wavelength)*.1 + (p0[0]+p0[-1])/(2.*fov)
    plt.plot(xw,yw,'b')
    ## 'xs' is a vector of x cordinates on the screen
    xs = np.arange(-1, 1, .01) 
    ## xsdiff is corresponding pathlength difference
    xsdiff = (np.sqrt(d2**2 + (xs-d3)**2) - np.sqrt(d2**2 + (xs+d3)**2))
    ## and we accumulate the interference pattern for each source into 'pattern'
    pattern = 0
    total_intensity = 0
    ## compute contribution to pattern from each source position p
    for pos,ampl in np.broadcast(p0,a0):
        total_intensity += ampl
        pos1 = pos/float(fov)
        if extent:  # simulate extent by plotting 100 sources of 1/100th intensity
            positions = np.arange(-1,1.01,.01)*extent/fov + pos1 
        else:
            positions = [pos1]
        # draw arrows indicating lightpath
        plt.arrow(-d1, bl2+pos1, d1, -pos1, head_width=.1, fc='k', length_includes_head=True)
        plt.arrow(-d1,-bl2+pos1, d1, -pos1, head_width=.1, fc='k', length_includes_head=True)
        for p in positions:
            # compute the pathlength difference between slits and position on screen
            plt.plot(-d1, p, marker='o', ms=10*ampl, mfc='red', mew=0)
            # add pathlength difference at slits
            diff = xsdiff + (baseline*wavelength)*np.sin(p*fov*np.pi/180)
            # accumulate interference pattern from this source
            pattern = pattern + (float(ampl)/len(positions))*np.cos(2*np.pi*diff/wavelength) 
    maxint = maxint or total_intensity
    # add fake axis to interference pattern just to make it a "wide" image
    pattern_image = pattern[:,np.newaxis] + np.zeros(10)[np.newaxis,:]
    plt.imshow(pattern_image, extent=(d2,d2+1,-1,1), cmap=plt.gray(), vmin=-maxint, vmax=maxint)
    # make a plot of the interference pattern
    plt.plot(d2+1.5+pattern/(maxint*2), xs, 'r')
    plt.show()
    print("visibility (Imax-Imin)/(Imax+Imin): ",(pattern.max()-pattern.min())/(total_intensity*2))
# show patern for one source at 0
michelson(p0=[0])


在这个版本中，光源位置不再用屏幕上的几何坐标表示，而是用波前到达角表示；基线也改为以波长数计。这样更接近真实天文场景，因为天体通常位于远场，进入干涉仪的是近似平面波。下面两个交互小工具是对本节主线的补充：一个展示单源情况下可见度相位与位置的关系，另一个展示双源情况下可见度振幅如何随结构变化。


In [ ]:
# single source
interact(lambda position, intensity, baseline: 
             michelson(p0=[position], a0=[intensity], baseline=baseline, maxint=2),
         position=(-5,5,.01),intensity=(.2,1,.01),baseline=(10,100,.01)) and None

下面再来看同样的实验，只不过这次使用两个光源：


In [ ]:
interact(lambda position1,position2,intensity1,intensity2,baseline: 
            michelson(p0=[position1,position2], a0=[intensity1,intensity2], baseline=baseline, maxint=2),
         position1=(-5,5,.01), position2=(-5,5,.01), intensity1=(.2,1,.01), intensity2=(.2,1,.01),
         baseline=(10,100,.01)) and None

#### A.1 参宿四尺寸测量

最后保留一个历史重现实验。通过改变基线长度并观察条纹可见度如何消失，可以直观理解为什么 Michelson 和 Pease 能够用干涉法测出参宿四的角直径，也能更深刻地体会“振幅测尺度、相位测位置”这句总结并不是口号，而是确实可以在一个可操作的模型里复现出来。


In [ ]:
arcsec = 1/3600.
interact(lambda extent_arcsec, baseline: 
             michelson(p0=[0], a0=[1], extent=extent_arcsec*arcsec, maxint=1, 
                       baseline=baseline,fov=1*arcsec),
         extent_arcsec=(0,0.1,0.001), 
         baseline=(1e+4,1e+7,1e+4)
        ) and None

***

* 下一节： [1.10 单碟望远镜的局限与价值](1_10_limits_of_single_dishes.ipynb)
